# **기계학습 프로그래밍 개요**


-------------------

## **에이전트  체험**

In [ ]:
!pip install -q anthropic

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 15.1 MB/s eta 0:00:00


* **Tool사용 예**:

In [ ]:
import anthropic, ast, operator
from google.colab import userdata   # Colab 보안 비밀에서 키를 읽음 (코드에 키 노출 안 됨)

client = anthropic.Anthropic(api_key=userdata.get("ANTHROPIC_API_KEY"))

# 1) 에이전트에게 줄 도구 "설명서" — LLM은 이 설명만 보고 호출 여부를 판단함
tools = [{
    "name": "calc",
    "description": "사칙연산 수식을 정확히 계산한다. 숫자 계산이 필요하면 반드시 이 도구를 써라.",
    "input_schema": {"type": "object",
                     "properties": {"expr": {"type": "string", "description": "예: (37**2-1234)/3"}},
                     "required": ["expr"]}
}]

# 2) 실제 도구 구현 — eval 대신 ast로 사칙연산만 허용 (안전 버전)
OPS = {ast.Add: operator.add, ast.Sub: operator.sub, ast.Mult: operator.mul,
       ast.Div: operator.truediv, ast.Pow: operator.pow, ast.USub: operator.neg}
def calc(expr: str) -> str:
    def ev(n):
        if isinstance(n, ast.Constant): return n.value
        if isinstance(n, ast.BinOp):   return OPS[type(n.op)](ev(n.left), ev(n.right))
        if isinstance(n, ast.UnaryOp): return OPS[type(n.op)](ev(n.operand))
        raise ValueError("허용되지 않은 수식")
    return str(ev(ast.parse(expr, mode="eval").body))

# 3) 에이전트 루프: 도구 호출이 끝날 때까지 반복
question = "37의 제곱에서 1234를 빼고 3으로 나눈 값은? 소수점 둘째 자리까지."
messages = [{"role": "user", "content": question}]

for step in range(5):                                   # 무한 루프 방지 상한
    res = client.messages.create(model="claude-sonnet-5", max_tokens=500,
                                 tools=tools, messages=messages)
    messages.append({"role": "assistant", "content": res.content})

    if res.stop_reason != "tool_use":                   # 도구가 더 필요 없으면 종료
        print(f"\n[{step+1}단계] 최종 답:", "".join(b.text for b in res.content if b.type == "text"))
        break

    results = []
    for blk in res.content:                             # 도구 호출 → 실행 → 결과 반환
        if blk.type == "tool_use":
            out = calc(blk.input["expr"])
            print(f"[{step+1}단계] 🔧 도구 호출: {blk.name}({blk.input['expr']}) → {out}")
            results.append({"type": "tool_result", "tool_use_id": blk.id, "content": out})
    messages.append({"role": "user", "content": results})

[1단계] 🔧 도구 호출: calc((37**2-1234)/3) → 45.0

[2단계] 최종 답: 37의 제곱은 1369이고, 여기서 1234를 빼면 135, 이를 3으로 나누면 **45.00**입니다.


* **No Tool 예**:

In [ ]:
# 도구 없이 같은 질문 → 자릿수 큰 계산으로 바꿔서 오차를 확인
q = "123456789 * 987654321 은?"
no_tool = client.messages.create(model="claude-sonnet-5", max_tokens=200,
                                 messages=[{"role":"user","content":q}])
print("도구 없음:", no_tool.content[0].text)
print("정답     :", 123456789 * 987654321)



---



## **머신러닝 vs 딥러닝**

* Machine Learning = 사람이 Feature 설계 + 모델이 분류 방법 학습
* Deep Learning = Feature Representation + Classifier 모두 함께 학


| 단계        | 전통적 머신러닝                   | 딥러닝                   |
| --------- | -------------------------- | --------------------- |
| **입력**    | 숫자 이미지(MNIST)              | 숫자 이미지(MNIST)         |
| **특징 추출** | **사람이 특징을 설계**             | **신경망이 특징을 학습**       |
| **예**     | 윤곽선(Edge), HOG, 평균 밝기 등    | CNN Filter가 자동 학습     |
| **분류**    | Logistic Regression, SVM 등 | Fully Connected Layer |
| **학습 대상** | 주로 **분류기의 가중치**            | **특징 추출기 + 분류기 모두**   |
| **코드 핵심** | `extract_features()`       | `Conv2d()`            |
| **출력**    | **3 / Not 3**              | **3 / Not 3**         |


### **예제: 숫자 3 / Not 3**
- **이미지 준비**: MNIST에서 숫자 이미지 자동 다운로드
- **숫자 판별**: 3 / Not 3을 구분하기
- **학습 데이터**: 60000장
    - **Train** : 2,000장
    - **Test**  :   400장

#### **1. 머신러닝 방식: 사람이 Feature를 설계**

In [ ]:
pip install torch torchvision opencv-python scikit-learn numpy

In [ ]:
# ============================================================
# Machine Learning 방식
#
# MNIST 자동 다운로드
#        ↓
# 사람이 Feature 설계 (extract_features())
#   - Edge Density
#   - HOG
#        ↓
# Logistic Regression
#        ↓
# 숫자 3 / Not 3
# ============================================================

import cv2
import numpy as np

from torchvision.datasets import MNIST

from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report


# ============================================================
# 1. 기본 설정
# ============================================================

DATA_PATH = "./data"

TARGET_NUMBER = 3

TRAIN_PER_CLASS = 1000
TEST_PER_CLASS = 200

RANDOM_SEED = 42


# ============================================================
# 2. MNIST 자동 다운로드
# ============================================================

print("MNIST 데이터를 다운로드합니다.")

train_dataset = MNIST(
    root=DATA_PATH,
    train=True,
    download=True
)

test_dataset = MNIST(
    root=DATA_PATH,
    train=False,
    download=True
)

print("Train 전체 데이터 :", len(train_dataset))
print("Test 전체 데이터  :", len(test_dataset))


# ============================================================
# 3. 3 / Not 3 데이터 균형 선택
# ============================================================

def get_balanced_indices(
        targets,
        target_number,
        samples_per_class,
        seed=42):

    targets = np.array(targets)

    # --------------------------------------------
    # 숫자 3
    # --------------------------------------------

    target_indices = np.where(
        targets == target_number
    )[0]


    # --------------------------------------------
    # 3이 아닌 숫자
    # --------------------------------------------

    not_target_indices = np.where(
        targets != target_number
    )[0]


    rng = np.random.default_rng(seed)


    # --------------------------------------------
    # 각각 같은 개수만 선택
    # --------------------------------------------

    selected_target = rng.choice(
        target_indices,
        size=samples_per_class,
        replace=False
    )

    selected_not_target = rng.choice(
        not_target_indices,
        size=samples_per_class,
        replace=False
    )


    # --------------------------------------------
    # 합치기
    # --------------------------------------------

    indices = np.concatenate([
        selected_target,
        selected_not_target
    ])


    # 순서 섞기
    rng.shuffle(indices)

    return indices


train_indices = get_balanced_indices(
    train_dataset.targets,
    TARGET_NUMBER,
    TRAIN_PER_CLASS,
    seed=42
)

test_indices = get_balanced_indices(
    test_dataset.targets,
    TARGET_NUMBER,
    TEST_PER_CLASS,
    seed=43
)


print("\n사용할 데이터")
print("Train :", len(train_indices))
print("Test  :", len(test_indices))


# ============================================================
# 4. HOG 생성
#
# 사람이
# "숫자의 모양과 방향 정보가 중요할 것이다."
# 라고 판단하여 HOG 특징을 사용
# ============================================================

hog = cv2.HOGDescriptor(
    _winSize=(28, 28),
    _blockSize=(14, 14),
    _blockStride=(7, 7),
    _cellSize=(7, 7),
    _nbins=9
)


# ============================================================
# 5. Feature Extraction
#
# 여기서 사람이 특징 추출 방법을 직접 결정한다.
# ============================================================

def extract_features(image):

    # numpy uint8 변환
    image = np.array(image, dtype=np.uint8)


    # --------------------------------------------------------
    # Feature 1 : Edge
    #
    # 사람이:
    # "숫자의 윤곽선이 중요할 것이다."
    # 라고 판단
    # --------------------------------------------------------
    edges = cv2.Canny(
        image,
        50,
        150
    )

    # Edge 비율
    edge_density = np.mean(
        edges > 0
    )

    # --------------------------------------------------------
    # Feature 2 : HOG
    #
    # 사람이:
    # "선의 방향과 숫자 모양이 중요할 것이다."
    # 라고 판단
    # --------------------------------------------------------
    hog_features = hog.compute(
        image
    ).flatten()


    # --------------------------------------------------------
    # Feature 3 : 평균 밝기
    #
    # 사람이 직접 추가한 특징
    # --------------------------------------------------------
    mean_brightness = np.mean(
        image
    ) / 255.0


    # --------------------------------------------------------
    # 모든 Feature 결합
    # --------------------------------------------------------
    manual_features = np.array([
        edge_density,
        mean_brightness
    ])


    features = np.concatenate([
        manual_features,
        hog_features
    ])

    return features


# ============================================================
# 6. Feature Dataset 생성
# ============================================================
def make_feature_dataset(
        dataset,
        indices):

    X = []
    y = []

    for index in indices:

        image, original_label = dataset[index]


        # --------------------------------------------
        # 사람이 만든 Feature Extraction
        # --------------------------------------------

        features = extract_features(
            image
        )


        X.append(features)


        # --------------------------------------------
        # 숫자 3 → 1
        # Not 3   → 0
        # --------------------------------------------

        if original_label == TARGET_NUMBER:

            y.append(1)

        else:

            y.append(0)


    return (
        np.array(X),
        np.array(y)
    )


print("\nTrain Feature 추출 중...")

X_train, y_train = make_feature_dataset(
    train_dataset,
    train_indices
)


print("Test Feature 추출 중...")

X_test, y_test = make_feature_dataset(
    test_dataset,
    test_indices
)


print("\nFeature Shape")
print("X_train :", X_train.shape)
print("X_test  :", X_test.shape)


# ============================================================
# 7. Machine Learning 모델
# ============================================================

model = make_pipeline(

    StandardScaler(),

    LogisticRegression(
        max_iter=1000
    )
)


# ============================================================
# 8. 학습
# ============================================================

print("\n==============================")
print("Machine Learning 학습")
print("==============================")

model.fit(
    X_train,
    y_train
)


# ============================================================
# 9. 예측
# ============================================================

prediction = model.predict(
    X_test
)


# ============================================================
# 10. 평가
# ============================================================

accuracy = accuracy_score(
    y_test,
    prediction
)


print("\n==============================")
print("Machine Learning 결과")
print("==============================")

print(
    f"Accuracy : {accuracy:.4f}"
)

print()

print(
    classification_report(
        y_test,
        prediction,
        target_names=[
            "Not 3",
            "3"
        ]
    )
)

MNIST 데이터를 다운로드합니다.


100%|██████████| 9.91M/9.91M [00:00<00:00, 54.0MB/s]
100%|██████████| 28.9k/28.9k [00:00<00:00, 1.74MB/s]
100%|██████████| 1.65M/1.65M [00:00<00:00, 14.7MB/s]
100%|██████████| 4.54k/4.54k [00:00<00:00, 10.2MB/s]
/tmp/ipykernel_1792/224909340.py:72: DeprecationWarning: __array__ implementation doesn't accept a copy keyword, so passing copy=False failed. __array__ must implement 'dtype' and 'copy' keyword arguments.
  targets = np.array(targets)


Train 전체 데이터 : 60000
Test 전체 데이터  : 10000

사용할 데이터
Train : 2000
Test  : 400

Train Feature 추출 중...
Test Feature 추출 중...

Feature Shape
X_train : (2000, 326)
X_test  : (400, 326)

Machine Learning 학습

Machine Learning 결과
Accuracy : 0.9925

              precision    recall  f1-score   support

       Not 3       0.99      0.99      0.99       200
           3       0.99      0.99      0.99       200

    accuracy                           0.99       400
   macro avg       0.99      0.99      0.99       400
weighted avg       0.99      0.99      0.99       400



#### **2. 딥러닝 방식: CNN이 Feature까지 학습**

In [ ]:
# ============================================================
# Deep Learning 방식
#
# MNIST 자동 다운로드
#        ↓
# CNN (Conv2d())
# Feature Learning + Classification
#        ↓
# 숫자 3 / Not 3
#
# 사람이 HOG, Edge를 직접 만들지 않는다.
# ============================================================

import numpy as np

import torch
import torch.nn as nn

from torch.utils.data import Dataset
from torch.utils.data import DataLoader

from torchvision.datasets import MNIST
from torchvision import transforms


# ============================================================
# 1. 기본 설정
# ============================================================

DATA_PATH = "./data"

TARGET_NUMBER = 3

TRAIN_PER_CLASS = 1000
TEST_PER_CLASS = 200

BATCH_SIZE = 64
EPOCHS = 5

RANDOM_SEED = 42


# ============================================================
# 2. Random Seed
# ============================================================

torch.manual_seed(
    RANDOM_SEED
)

np.random.seed(
    RANDOM_SEED
)


# ============================================================
# 3. CPU / GPU
# ============================================================

device = torch.device(

    "cuda"

    if torch.cuda.is_available()

    else "cpu"
)

print(
    "사용 Device :",
    device
)


# ============================================================
# 4. 이미지 → Tensor
#
# 이것은 Feature Engineering이 아니다.
#
# 단순히 신경망 입력 형식으로 변환하는 전처리이다.
# ============================================================

transform = transforms.Compose([

    transforms.ToTensor(),

    transforms.Normalize(
        (0.5,),
        (0.5,)
    )

])


# ============================================================
# 5. MNIST 자동 다운로드
# ============================================================

print("\nMNIST 데이터를 다운로드합니다.")

train_base = MNIST(
    root=DATA_PATH,
    train=True,
    download=True,
    transform=transform
)

test_base = MNIST(
    root=DATA_PATH,
    train=False,
    download=True,
    transform=transform
)


print(
    "Train 전체 데이터 :",
    len(train_base)
)

print(
    "Test 전체 데이터  :",
    len(test_base)
)


# ============================================================
# 6. 동일하게 3 / Not 3 균형 데이터 선택
# ============================================================

def get_balanced_indices(
        targets,
        target_number,
        samples_per_class,
        seed=42):

    targets = np.array(
        targets
    )


    # 숫자 3
    target_indices = np.where(
        targets == target_number
    )[0]


    # 3이 아닌 숫자
    not_target_indices = np.where(
        targets != target_number
    )[0]


    rng = np.random.default_rng(
        seed
    )


    selected_target = rng.choice(

        target_indices,

        size=samples_per_class,

        replace=False
    )


    selected_not_target = rng.choice(

        not_target_indices,

        size=samples_per_class,

        replace=False
    )


    indices = np.concatenate([

        selected_target,

        selected_not_target
    ])


    rng.shuffle(
        indices
    )


    return indices


train_indices = get_balanced_indices(
    train_base.targets,
    TARGET_NUMBER,
    TRAIN_PER_CLASS,
    seed=42
)

test_indices = get_balanced_indices(
    test_base.targets,
    TARGET_NUMBER,
    TEST_PER_CLASS,
    seed=43
)


print("\n사용할 데이터")
print(
    "Train :",
    len(train_indices)
)

print(
    "Test  :",
    len(test_indices)
)


# ============================================================
# 7. Binary Dataset
#
# 숫자 3 → 1
# Not 3   → 0
# ============================================================

class BinaryMNISTDataset(Dataset):

    def __init__(
            self,
            base_dataset,
            indices):

        self.base_dataset = (
            base_dataset
        )

        self.indices = (
            indices
        )


    def __len__(self):

        return len(
            self.indices
        )


    def __getitem__(
            self,
            index):

        original_index = (
            self.indices[index]
        )


        image, original_label = (
            self.base_dataset[
                original_index
            ]
        )


        # --------------------------------------------
        # Binary Label
        # --------------------------------------------

        if original_label == TARGET_NUMBER:

            label = 1

        else:

            label = 0


        return (
            image,
            label
        )


train_dataset = BinaryMNISTDataset(
    train_base,
    train_indices
)

test_dataset = BinaryMNISTDataset(
    test_base,
    test_indices
)


# ============================================================
# 8. DataLoader
# ============================================================

train_loader = DataLoader(

    train_dataset,

    batch_size=BATCH_SIZE,

    shuffle=True
)


test_loader = DataLoader(

    test_dataset,

    batch_size=BATCH_SIZE,

    shuffle=False
)


# ============================================================
# 9. CNN 모델
# ============================================================

class NumberCNN(nn.Module):

    def __init__(self):

        super().__init__()


        # ====================================================
        # Feature Extraction
        #
        # 사람이
        #
        # Edge
        # HOG
        #
        # 를 직접 만들지 않는다.
        #
        # Conv2d의 Filter Weight가
        # Backpropagation으로 학습된다.
        # ====================================================

        self.features = nn.Sequential(


            # --------------------------------------------
            # 입력
            #
            # 1 × 28 × 28
            # --------------------------------------------

            nn.Conv2d(
                in_channels=1,
                out_channels=16,
                kernel_size=3,
                padding=1
            ),

            nn.ReLU(),

            nn.MaxPool2d(2),


            # --------------------------------------------
            # 16 × 14 × 14
            # --------------------------------------------

            nn.Conv2d(
                in_channels=16,
                out_channels=32,
                kernel_size=3,
                padding=1
            ),

            nn.ReLU(),

            nn.MaxPool2d(2)

            # 최종:
            # 32 × 7 × 7
        )


        # ====================================================
        # Classifier
        # ====================================================

        self.classifier = nn.Sequential(

            nn.Flatten(),

            nn.Linear(
                32 * 7 * 7,
                64
            ),

            nn.ReLU(),

            nn.Linear(
                64,
                2
            )
        )


    def forward(self, x):

        # Feature Learning
        x = self.features(x)

        # Classification
        x = self.classifier(x)

        return x


# ============================================================
# 10. 모델 생성
# ============================================================

model = NumberCNN().to(
    device
)


print("\nCNN 모델")

print(
    model
)


# ============================================================
# 11. 학습 전 CNN Filter 저장
#
# 나중에 비교해서
# Conv Filter 자체가 학습되었음을 확인한다.
# ============================================================

before_filter = (

    model.features[0]
    .weight[0, 0]
    .detach()
    .cpu()
    .clone()
)


print(
    "\n학습 전 첫 번째 CNN Filter"
)

print(
    before_filter
)


# ============================================================
# 12. Loss
# ============================================================

criterion = nn.CrossEntropyLoss()


# ============================================================
# 13. Optimizer
#
# model.parameters()에는
#
# Conv Layer Weight
# Linear Layer Weight
#
# 모두 포함되어 있다.
# ============================================================

optimizer = torch.optim.Adam(

    model.parameters(),

    lr=0.001
)


# ============================================================
# 14. 학습
# ============================================================

print("\n==============================")
print("Deep Learning 학습 시작")
print("==============================")


for epoch in range(EPOCHS):

    model.train()

    total_loss = 0

    correct = 0

    total = 0


    for images, labels in train_loader:

        images = images.to(
            device
        )

        labels = labels.to(
            device
        )


        # --------------------------------------------
        # Gradient 초기화
        # --------------------------------------------

        optimizer.zero_grad()


        # --------------------------------------------
        # Forward
        #
        # Feature Extraction
        # +
        # Classification
        # --------------------------------------------

        outputs = model(
            images
        )


        # --------------------------------------------
        # Loss
        # --------------------------------------------

        loss = criterion(
            outputs,
            labels
        )


        # --------------------------------------------
        # Backpropagation
        #
        # Conv Filter까지 Gradient 전달
        # --------------------------------------------

        loss.backward()


        # --------------------------------------------
        # Weight Update
        #
        # Feature Extractor
        # +
        # Classifier
        # 모두 업데이트
        # --------------------------------------------

        optimizer.step()


        total_loss += (
            loss.item()
        )


        _, predicted = torch.max(
            outputs,
            1
        )


        total += labels.size(0)


        correct += (
            predicted == labels
        ).sum().item()


    accuracy = (
        correct / total
    )


    avg_loss = (

        total_loss

        / len(train_loader)
    )


    print(

        f"Epoch "
        f"{epoch + 1:2d}/{EPOCHS}"

        f" | Loss = "
        f"{avg_loss:.4f}"

        f" | Accuracy = "
        f"{accuracy:.4f}"
    )


# ============================================================
# 15. Test
# ============================================================

model.eval()

correct = 0
total = 0


with torch.no_grad():

    for images, labels in test_loader:

        images = images.to(
            device
        )

        labels = labels.to(
            device
        )


        outputs = model(
            images
        )


        _, predicted = torch.max(
            outputs,
            1
        )


        total += labels.size(0)


        correct += (
            predicted == labels
        ).sum().item()


test_accuracy = (
    correct / total
)


print("\n==============================")
print("Deep Learning 결과")
print("==============================")


print(
    f"Test Accuracy : "
    f"{test_accuracy:.4f}"
)


# ============================================================
# 16. 학습 후 CNN Filter
# ============================================================

after_filter = (

    model.features[0]
    .weight[0, 0]
    .detach()
    .cpu()
    .clone()
)


print(
    "\n학습 후 첫 번째 CNN Filter"
)

print(
    after_filter
)


# ============================================================
# 17. CNN Filter가 실제로 변했는지 확인
# ============================================================

difference = (

    after_filter

    - before_filter
)


print(
    "\nCNN Filter Weight 변화량"
)

print(
    difference
)


print(
    "\nFilter 변화량 평균 절댓값 :",
    difference.abs().mean().item()
)

사용 Device : cpu

MNIST 데이터를 다운로드합니다.
Train 전체 데이터 : 60000
Test 전체 데이터  : 10000


/tmp/ipykernel_1792/3771231883.py:137: DeprecationWarning: __array__ implementation doesn't accept a copy keyword, so passing copy=False failed. __array__ must implement 'dtype' and 'copy' keyword arguments.
  targets = np.array(



사용할 데이터
Train : 2000
Test  : 400

CNN 모델
NumberCNN(
  (features): Sequential(
    (0): Conv2d(1, 16, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (1): ReLU()
    (2): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (3): Conv2d(16, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (4): ReLU()
    (5): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  )
  (classifier): Sequential(
    (0): Flatten(start_dim=1, end_dim=-1)
    (1): Linear(in_features=1568, out_features=64, bias=True)
    (2): ReLU()
    (3): Linear(in_features=64, out_features=2, bias=True)
  )
)

학습 전 첫 번째 CNN Filter
tensor([[ 0.2548,  0.2767, -0.0781],
        [ 0.3062, -0.0730,  0.0673],
        [-0.1623,  0.1958,  0.2938]])

Deep Learning 학습 시작
Epoch  1/5 | Loss = 0.5543 | Accuracy = 0.7645
Epoch  2/5 | Loss = 0.2374 | Accuracy = 0.9155
Epoch  3/5 | Loss = 0.1687 | Accuracy = 0.9480
Epoch  4/5 | Loss = 0.1382 | Accuracy = 0.9560
Epoch  5/5 |



---



## AI가 잘하는 것 못하는 것

In [ ]:
!pip install -q google-genai

In [ ]:
from google import genai
from google.colab import userdata
client = genai.Client(api_key=userdata.get("GEMINI_API_KEY"))

tests = [
  ("요약",     "다음 문단을 한 문장으로: 인공지능은 ...", "잘함"),
  ("산수",     "17389 곱하기 4213 은? 계산 과정 없이 답만.", "약함"),
  ("최신정보", "오늘 서울의 기온은?", "모름"),
  ("글자수",   "'딸기'라는 단어가 들어간 12글자 문장을 만들어줘.", "약함"),
]

for name, q, expect in tests:
    r = client.interactions.create(model="gemini-3-flash-preview", input=q)
    print(f"[{name}] 예상={expect}")
    print(" →", r.output_text.strip()[:200], "\n")

# 같은 질문에 thinking level을 높이면 산수 정확도가 올라가는지 비교해 보자

InternalServerError: Error code: 500 - {'error': {'message': 'gemini-3-flash-preview is currently experiencing high demand, spikes in demand are usually temporary. Please try again later.', 'code': 'api_error'}}